# 06_mllm / gemini_setup.ipynb

**Purpose:** One-time setup for Gemini 2.5 Flash-Lite — used to generate human-readable explanations of GSAF predictions.

**Why Gemini direct (not OpenRouter):**
- OpenRouter free tier = 50 requests/day total. With our 1000-call budget that is 20 days.
- **Gemini API direct from Google AI Studio** = 1000 RPD on Flash-Lite, free, no credit card.
- 1000 calls = 500 free-form + 500 CoT = whole job in 1 day, well inside your 2-day budget.

**Free-tier limits (Gemini 2.5 Flash-Lite, April 2026):**
| Model | RPM | RPD |
|---|---|---|
| **Gemini 2.5 Flash-Lite** (primary) | 15 | **1000** |
| Gemini 2.5 Flash (fallback) | 10 | 250 |

**What this notebook does:**
1. Installs `google-generativeai` if missing.
2. Loads `GEMINI_API_KEY` from `.env` (or prompts for it).
3. Smoke-tests text and vision.
4. Writes the helper module `src/utils/mllm_client.py` (used by the other two notebooks).
5. Writes a sentinel file so downstream notebooks know setup ran.

**Inputs:**
- API key from <https://aistudio.google.com/apikey> (free, no credit card).
- Add to `E:\eve-rag\.env` as `GEMINI_API_KEY=AIzaSy...`

**Outputs:**
- `src/utils/mllm_client.py` — helper module
- `cache/mllm/.mllm_setup_ok` — sentinel
- `cache/mllm/gemini_responses/` — disk cache (filled by next two notebooks)


In [1]:
# 1. Install dependencies (idempotent)
import subprocess, sys
def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        print(f"Installing {pkg}…")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("google-generativeai", "google.generativeai")
_ensure("python-dotenv", "dotenv")
_ensure("Pillow", "PIL")

import google.generativeai as genai
print("google-generativeai version:", getattr(genai, "__version__", "ok"))


google-generativeai version: 0.8.6


C:\Users\User\AppData\Local\Temp\ipykernel_21044\2853913584.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  __import__(import_name or pkg)


In [2]:
# 2. Path setup
from __future__ import annotations
import os, sys, json, time
from pathlib import Path

NB_DIR = Path.cwd()
ROOT = NB_DIR
for parent in [NB_DIR] + list(NB_DIR.parents):
    if (parent / "src" / "utils" / "paths.py").exists():
        ROOT = parent
        break
sys.path.insert(0, str(ROOT))

from src.utils.paths import ALL_MANIFEST, PROCESSED_IMAGES, VERITE_IMAGES

CACHE_MLLM = ROOT / "cache" / "mllm"
CACHE_MLLM.mkdir(parents=True, exist_ok=True)
RESPONSE_CACHE = CACHE_MLLM / "gemini_responses"
RESPONSE_CACHE.mkdir(parents=True, exist_ok=True)

ENV_PATH = ROOT / ".env"
print("Repo root:", ROOT)
print("MLLM cache:", CACHE_MLLM)
print(".env exists:", ENV_PATH.exists())


Repo root: e:\eve-rag
MLLM cache: e:\eve-rag\cache\mllm
.env exists: True


## 3. Load API key

If you do not have one yet:

1. Go to <https://aistudio.google.com/apikey> (sign in with your Google account).
2. Click **Create API key** → copy it.
3. Add to `E:\eve-rag\.env`:
   ```
   GEMINI_API_KEY=AIzaSy...
   ```
4. Re-run this cell.


In [3]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=ENV_PATH, override=True)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()
if not GEMINI_API_KEY:
    GEMINI_API_KEY = input("Paste your GEMINI_API_KEY (won't be saved to disk): ").strip()

assert GEMINI_API_KEY, "No API key. Get one free at https://aistudio.google.com/apikey"
assert GEMINI_API_KEY.startswith("AIza"), "API key should start with 'AIza' for Gemini."
print(f"API key loaded (length={len(GEMINI_API_KEY)}, prefix={GEMINI_API_KEY[:6]}…)")


API key loaded (length=39, prefix=AIzaSy…)


## 4. Configure SDK and pick model

In [4]:
import google.generativeai as genai
genai.configure(api_key=GEMINI_API_KEY)

# Find available models — pick Flash-Lite (1000 RPD) primarily, Flash as fallback
PREFERRED = [
    "models/gemini-2.5-flash-lite",      # 15 RPM / 1000 RPD — primary
    "models/gemini-2.5-flash",            # 10 RPM /  250 RPD — fallback
    "models/gemini-2.0-flash",            # legacy fallback
    "models/gemini-1.5-flash",            # legacy fallback
]

available = []
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        available.append(m.name)

print(f"Found {len(available)} generateContent-capable models on your account.")

MODEL_NAME = None
for cand in PREFERRED:
    if cand in available:
        MODEL_NAME = cand
        break

# If none of the preferred match (e.g. naming change), fall back to first flash-lite/flash
if MODEL_NAME is None:
    for cand in available:
        if "flash-lite" in cand: MODEL_NAME = cand; break
if MODEL_NAME is None:
    for cand in available:
        if "flash" in cand: MODEL_NAME = cand; break

assert MODEL_NAME is not None, f"No flash model available. Available: {available[:10]}"
print("Selected:", MODEL_NAME)


Found 38 generateContent-capable models on your account.
Selected: models/gemini-2.5-flash-lite


## 5. Smoke test 1 — text only

In [5]:
model = genai.GenerativeModel(MODEL_NAME)
resp = model.generate_content("Reply with exactly: 'EVE-RAG setup OK'")
reply = (resp.text or "").strip()
print("Text reply:", reply)
assert "EVE-RAG" in reply or "OK" in reply, f"Unexpected text reply: {reply!r}"


Text reply: EVE-RAG setup OK


## 6. Smoke test 2 — image + caption

In [6]:
import pandas as pd
from PIL import Image

m = pd.read_csv(ALL_MANIFEST)
candidates = m[(m["has_image"] == 1) & (m["dataset"].isin(["mmfakebench", "verite"]))].head(50)

test_row, test_img = None, None
for _, r in candidates.iterrows():
    p = Path(str(r.get("image_path") or ""))
    if not p.is_absolute():
        for base in [PROCESSED_IMAGES, VERITE_IMAGES, ROOT]:
            cand = base / p.name
            if cand.exists(): p = cand; break
            cand2 = base / p
            if cand2.exists(): p = cand2; break
    if p.exists():
        test_row, test_img = r, p; break

assert test_row is not None, "No local image found for smoke test."
print("Test image:", test_img)
print("Caption   :", str(test_row["caption"])[:140])
print("Label     :", test_row["label"])

img = Image.open(test_img).convert("RGB")
img.thumbnail((512, 512))

prompt = (
    "You are an analyst checking whether an image and a caption match. "
    "In one sentence, describe what is shown in the image. "
    "In one more sentence, say whether the image plausibly matches the caption: "
    f"'{test_row['caption']}'."
)

t0 = time.time()
resp = model.generate_content([prompt, img])
dt = time.time() - t0
print(f"\nReply ({dt:.1f}s):")
print((resp.text or "").strip())


C:\Users\User\AppData\Local\Temp\ipykernel_21044\4037449554.py:4: DtypeWarning: Columns (0: image_path, 1: source_image_id, 2: text_source, 3: image_source, 4: fake_cls, 5: raw_label) have mixed types. Specify dtype option on import or set low_memory=False.
  m = pd.read_csv(ALL_MANIFEST)


Test image: E:\eve-rag\data\processed\ooc_pairs\images\verite_true_0.jpg
Caption   : Photograph shows a family that was taken away by local epidemic prevention personnel after they had returned home from Hubei Province in Feb
Label     : 0

Reply (5.3s):
The image shows several people in hazmat suits escorting an individual down a hallway. The image plausibly matches the caption, as it depicts individuals in protective gear handling a person in what could be a quarantine or epidemic prevention scenario.


## 7. Write helper module `src/utils/mllm_client.py`

In [7]:
HELPER_PATH = ROOT / "src" / "utils" / "mllm_client.py"

helper_code = '"""\nGemini 2.5 Flash-Lite client for EVE-RAG.\n\nUsed by:\n    notebooks/06_mllm/mllm_inference.ipynb\n    notebooks/06_mllm/chain_of_thought.ipynb\n\nFeatures:\n- Disk cache keyed by hash(prompt + image bytes + model). Free re-runs.\n- Soft RPM rate limit (default 14, headroom under 15 RPM Flash-Lite limit).\n- Daily request counter so we stop cleanly before hitting 1000 RPD.\n- Retries on 429 / 500 / 503 with exponential backoff.\n- Falls back from Flash-Lite to Flash (if you set fallback_model).\n"""\nfrom __future__ import annotations\nimport os, json, time, hashlib\nfrom pathlib import Path\nfrom dataclasses import dataclass, field\nfrom datetime import date\n\nimport google.generativeai as genai\nfrom PIL import Image\nfrom dotenv import load_dotenv\n\n\n@dataclass\nclass MLLMClient:\n    model_name: str = "models/gemini-2.5-flash-lite"\n    fallback_model: str | None = "models/gemini-2.5-flash"\n    rpm_limit: int = 14            # headroom under 15 RPM\n    rpd_limit: int = 950           # headroom under 1000 RPD\n    cache_dir: Path | None = None\n    image_max_side: int = 512\n    max_retries: int = 5\n    request_log: list = field(default_factory=list)\n\n    def __post_init__(self):\n        here = Path(__file__).resolve()\n        for p in [here] + list(here.parents):\n            if (p / "src" / "utils" / "paths.py").exists():\n                self.repo_root = p; break\n        else:\n            self.repo_root = Path.cwd()\n\n        load_dotenv(self.repo_root / ".env", override=True)\n        api_key = os.environ.get("GEMINI_API_KEY", "").strip()\n        if not api_key:\n            raise RuntimeError("GEMINI_API_KEY not in .env (repo root).")\n        genai.configure(api_key=api_key)\n\n        self._primary = genai.GenerativeModel(self.model_name)\n        self._fallback = genai.GenerativeModel(self.fallback_model) if self.fallback_model else None\n\n        if self.cache_dir is None:\n            self.cache_dir = self.repo_root / "cache" / "mllm" / "gemini_responses"\n        self.cache_dir.mkdir(parents=True, exist_ok=True)\n\n        # Daily counter: tiny json file keyed by today\'s date\n        self._counter_path = self.repo_root / "cache" / "mllm" / "daily_counter.json"\n        self._load_counter()\n\n    # ------- counter -------\n    def _load_counter(self):\n        today = str(date.today())\n        if self._counter_path.exists():\n            try:\n                d = json.loads(self._counter_path.read_text())\n                if d.get("date") == today:\n                    self._daily_count = int(d.get("count", 0))\n                    return\n            except Exception:\n                pass\n        self._daily_count = 0\n        self._save_counter()\n\n    def _save_counter(self):\n        self._counter_path.write_text(json.dumps({\n            "date": str(date.today()),\n            "count": self._daily_count,\n        }))\n\n    def _bump_counter(self):\n        self._daily_count += 1\n        self._save_counter()\n\n    def daily_remaining(self) -> int:\n        return max(0, self.rpd_limit - self._daily_count)\n\n    # ------- cache key -------\n    @staticmethod\n    def _key(prompt: str, image_path, model_name: str) -> str:\n        h = hashlib.sha256()\n        h.update(model_name.encode()); h.update(b"||")\n        h.update(prompt.encode("utf-8")); h.update(b"||")\n        if image_path is not None and Path(image_path).exists():\n            h.update(Path(image_path).read_bytes())\n        return h.hexdigest()[:32]\n\n    # ------- rate limit -------\n    def _wait_if_needed(self):\n        now = time.time()\n        self.request_log = [t for t in self.request_log if now - t < 60]\n        if len(self.request_log) >= self.rpm_limit:\n            sleep_for = 60 - (now - self.request_log[0]) + 0.5\n            if sleep_for > 0:\n                time.sleep(sleep_for)\n                self.request_log = [t for t in self.request_log if time.time() - t < 60]\n\n    # ------- main API -------\n    def generate(self, prompt: str, image_path=None,\n                 use_cache: bool = True, temperature: float = 0.2,\n                 max_output_tokens: int = 512) -> str:\n        """Return Gemini text response. Cached, rate-limited, restartable."""\n        key = self._key(prompt, image_path, self.model_name)\n        cache_file = self.cache_dir / f"{key}.json"\n\n        if use_cache and cache_file.exists():\n            try:\n                return json.loads(cache_file.read_text(encoding="utf-8"))["text"]\n            except Exception:\n                pass\n\n        if self._daily_count >= self.rpd_limit:\n            raise RuntimeError(\n                f"Daily quota of {self.rpd_limit} reached. Resume tomorrow "\n                f"(quota resets at midnight Pacific). Cache is preserved."\n            )\n\n        self._wait_if_needed()\n        parts = [prompt]\n        if image_path is not None:\n            img = Image.open(image_path).convert("RGB")\n            img.thumbnail((self.image_max_side, self.image_max_side))\n            parts.append(img)\n\n        gen_cfg = {"temperature": temperature, "max_output_tokens": max_output_tokens}\n\n        last_err = None\n        models_to_try = [self._primary] + ([self._fallback] if self._fallback else [])\n        for mdl in models_to_try:\n            for attempt in range(self.max_retries):\n                try:\n                    self.request_log.append(time.time())\n                    resp = mdl.generate_content(parts, generation_config=gen_cfg)\n                    text = (resp.text or "").strip()\n                    cache_file.write_text(json.dumps({\n                        "text": text,\n                        "prompt": prompt[:2000],\n                        "image": str(image_path) if image_path else None,\n                        "model": mdl.model_name,\n                        "ts": time.time(),\n                    }), encoding="utf-8")\n                    self._bump_counter()\n                    return text\n                except Exception as e:\n                    last_err = e\n                    msg = str(e).lower()\n                    transient = any(k in msg for k in\n                        ("429", "quota", "rate", "deadline", "503", "500", "unavailable", "internal"))\n                    if transient:\n                        backoff = min(60.0, 2 ** attempt * 5)\n                        time.sleep(backoff)\n                        continue\n                    # Non-transient — try fallback model\n                    break\n        raise RuntimeError(f"Gemini call failed after retries (and fallback): {last_err}")\n'

HELPER_PATH.write_text(helper_code, encoding="utf-8")
print("Wrote helper:", HELPER_PATH)
print(f"({len(helper_code):,} chars)")


Wrote helper: e:\eve-rag\src\utils\mllm_client.py
(6,493 chars)


## 8. Verify helper module works end-to-end

In [8]:
import importlib
if "src.utils.mllm_client" in sys.modules:
    importlib.reload(sys.modules["src.utils.mllm_client"])
from src.utils.mllm_client import MLLMClient

mc = MLLMClient()
print("Helper loaded.")
print("  model_name      :", mc.model_name)
print("  fallback_model  :", mc.fallback_model)
print("  daily remaining :", mc.daily_remaining())

out = mc.generate(
    prompt="In one sentence, what does this image show?",
    image_path=test_img,
    use_cache=True,
    temperature=0.0,
    max_output_tokens=120,
)
print("\nHelper test reply:")
print(out)


Helper loaded.
  model_name      : models/gemini-2.5-flash-lite
  fallback_model  : models/gemini-2.5-flash
  daily remaining : 950

Helper test reply:
The image shows a woman being carried by people in hazmat suits down a hallway.


## 9. Write sentinel file

In [9]:
sentinel = CACHE_MLLM / ".mllm_setup_ok"
sentinel.write_text(json.dumps({
    "model_name":     MODEL_NAME,
    "fallback_model": "models/gemini-2.5-flash",
    "provider":       "gemini",
    "rpm_limit":      14,
    "rpd_limit":      950,
    "ts":             time.time(),
}, indent=2), encoding="utf-8")
print("Wrote sentinel:", sentinel)
print()
print("✓ Setup complete. Proceed to 06_mllm/mllm_inference.ipynb")


Wrote sentinel: e:\eve-rag\cache\mllm\.mllm_setup_ok

✓ Setup complete. Proceed to 06_mllm/mllm_inference.ipynb


---
**Done.** The next two notebooks (`mllm_inference.ipynb`, `chain_of_thought.ipynb`) import `MLLMClient` from `src.utils.mllm_client` and use it for batch generation.